# 02 - Cleaning, weather, features
Every raw row gets exactly one outcome; the exclusion report is the cleaning log.

In [ ]:
import os, sys, pathlib
MODE = "real"            # "sample" shows the SYNTHETIC test run (clearly watermarked). Never mix the two.
os.environ["DATA_MODE"] = MODE
# os.environ["START_MONTH"] = "2024-06"; os.environ["END_MONTH"] = "2024-06"
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import json, duckdb, pandas as pd
from IPython.display import Image, Markdown, HTML, display
import config, provenance
print(provenance.banner())
if config.IS_SYNTHETIC:
    display(HTML("<h3 style='color:red'>SYNTHETIC DATA - pipeline test only, not Citi Bike findings</h3>"))
man = provenance.load()
assert man is not None, "No manifest for this run yet - run the earlier phases first."

In [ ]:
import cleaning, weather, features
cleaning.main(); weather.main(); features.main()

In [ ]:
pd.read_csv(config.EXCLUSION_REPORT)

**How to read this (guide, not a finding):** `rows_parsed` minus all `excluded:` lines equals `FINAL rows kept`. `info (kept)` lines are NOT exclusions. A large `missing_end_coords` share means many trips have no distance and are excluded from the duration model only. `outside_requested_range` rows come from files containing trips just outside the month.

In [ ]:
F = f"read_parquet('{config.sql_path(config.TRIPS_FEAT)}')"
duckdb.sql(f"SELECT count(*) rows, min(started_at) first, max(started_at) last, avg(is_staff_trip::INT) staff_share, avg((temp_c IS NULL)::INT) no_weather FROM {F}").df()

In [ ]:
man['stages']['weather'], man['stages']['features']

**Checklist:** weather coverage ≈100%? no-weather share below 0.5%? coverage by month sensible (`man['stages']['cleaning']['coverage']['rows_by_month']`)?